In [2]:
!mamba install pandas


mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.641 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.0.6    np23py313h124dbc7_0  emscripten-forge-4x
+ python-tzdata  2026.5   pyh5ded981_0         conda-forge
- pip            26.2.1   pyh145f28c_0         conda-forge


In [3]:
import pandas as pd

data = ({
    "study_hours": [2, 3, 5, 1, 4, 2, 6, 3, 1, 4],
    "attendance": [70, 80, 90, 50, 85, 60, 95, 70, 40, 75],
    "previous_cgpa": [7.5, 8.0, 6.5, 6.0, 9.3, 9.0, 7.0, 6.0, 5.0, 6.0],
    "target_cgpa": [7.6, 8.2, 7.0, 6.1, 8.0, 8.7, 7.5, 6.5, 5.5, 6.8]
})

df = pd.DataFrame(data)

df.to_csv("student_performance.csv", index=False)

X = df[["study_hours", "attendance", "previous_cgpa"]]
y = df["target_cgpa"]

df.head()

,study_hours,attendance,previous_cgpa,target_cgpa
0,2,70,7.5,7.6
1,3,80,8.0,8.2
2,5,90,6.5,7.0
3,1,50,6.0,6.1
4,4,85,9.3,8.0


In [4]:
X_train = X.iloc[:8]
X_test = X.iloc[8:]

y_train = y.iloc[:8]
y_test = y.iloc[8:]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8, 3)
(2, 3)
(8,)
(2,)


In [5]:
baseline = y_train.mean()

print("Baseline CGPA prediction:", round(baseline, 2))
print("Actual target CGPA:", list(y_test))
print("Baseline predictions:", [round(baseline, 2)] * len(y_test))

Baseline CGPA prediction: 7.45
Actual target CGPA: [5.5, 6.8]
Baseline predictions: [np.float64(7.45), np.float64(7.45)]


In [6]:
import numpy as np

X_train_array = X_train.to_numpy(dtype=float)
y_train_array = y_train.to_numpy(dtype=float)

X_test_array = X_test.to_numpy(dtype=float)
y_test_array = y_test.to_numpy(dtype=float)

print("Training features:", X_train_array.shape)
print("Training targets:", y_train_array.shape)

Training features: (8, 3)
Training targets: (8,)


In [7]:
X_with_bias = np.column_stack([
    np.ones(len(X_train_array)),
    X_train_array
])

coefficients = np.linalg.lstsq(
    X_with_bias,
    y_train_array,
    rcond=None
)[0]

print("Model coefficients:", coefficients)

Model coefficients: [ 2.25556697 -0.04111877  0.01127608  0.60470328]


In [11]:
X_test_with_bias = np.column_stack([
    np.ones(len(X_test_array)),
    X_test_array
])

predictions = X_test_with_bias @ coefficients

# CGPA cannot be below 0 or above 10
predictions = np.clip(predictions, 0, 10)

print("Actual target CGPA:", y_test_array)
print("Predicted target CGPA:", np.round(predictions, 2))

Actual target CGPA: [5.5 6.8]
Predicted target CGPA: [5.69 6.57]


In [12]:
actual_mean = np.mean(y_test_array)

ss_res = np.sum((y_test_array - predictions) ** 2)
ss_tot = np.sum((y_test_array - actual_mean) ** 2)

if ss_tot == 0:
    print("R² score cannot be calculated for this test set.")
else:
    r2 = 1 - (ss_res / ss_tot)
    print("R² score:", round(r2, 3))

mae = np.mean(np.abs(y_test_array - predictions))
print("Mean Absolute Error:", round(mae, 2))

R² score: 0.892
Mean Absolute Error: 0.21


In [10]:
def predict_cgpa(study_hours, attendance, previous_cgpa):
    student = np.array([[
        study_hours,
        attendance,
        previous_cgpa ]])

    student_with_bias = np.column_stack([
        np.ones(len(student)),
        student])

    prediction = student_with_bias @ coefficients

    # Keep the prediction within the CGPA scale
    prediction = np.clip(prediction, 0, 10)

    return round(float(prediction[0]), 2)

In [21]:
study_hours = float(input("Study hours per day (0-24): "))
attendance = float(input("Attendance percentage (0-100): "))
previous_cgpa = float(input("Previous CGPA (0-10): "))

if (study_hours < 0 or study_hours > 24
    or attendance < 0 or attendance > 100
    or previous_cgpa < 0 or previous_cgpa > 10):
    
    print("Invalid input. Please check your values.")

else:
    result = predict_cgpa(
        study_hours,
        attendance,
        previous_cgpa )

print("Predicted target CGPA:", result, "/ 10")

if result>=8: 
    print("Excellent")
elif result>6: 
    print("Good!Keep it up")
else: 
    print("Needs improvemnt!Study more")

Study hours per day (0-24):  4
Attendance percentage (0-100):  80
Previous CGPA (0-10):  7.3


Predicted target CGPA: 7.41 / 10
Good!Keep it up
